# 🌿 PlantAI — Exploración, Augmentación y Evaluación

Este notebook presenta de forma visual tres aspectos clave del proyecto:

1. **Exploración del dataset** — distribución de clases, fuentes y ejemplos de imágenes
2. **Augmentación de datos** — comparación visual de los 3 niveles (leve · intensa · extrema)
3. **Evaluación del modelo** — accuracy, F1, top-5 y matriz de confusión

---
### Archivos necesarios en Google Drive

Sube los siguientes archivos a `Mi Unidad/PlantAI/` en tu Google Drive:

| Archivo | Descripción |
|---|---|
| `manifest.csv` | Índice de imágenes generado por `build_dataset.py` |
| `class_names.json` | Lista de las 114 clases |
| `plant_disease_model.keras` | Modelo entrenado |

> Los datasets de imágenes se descargan automáticamente desde Kaggle. No es necesario subir imágenes ni credenciales.

## ⚙️ 0 · Instalación y configuración

In [ ]:
# Instalar dependencias que Colab no incluye por defecto
!pip install -q seaborn scikit-learn kagglehub

In [ ]:
import os
import csv
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from pathlib import Path
from PIL import Image

print(f'TensorFlow {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Rutas en Drive -- ajusta DRIVE_ROOT si usaste otra carpeta
DRIVE_ROOT = '/content/drive/MyDrive/PlantAI'

MANIFEST_PATH = os.path.join(DRIVE_ROOT, 'manifest.csv')
CLASSES_PATH  = os.path.join(DRIVE_ROOT, 'class_names.json')
MODEL_PATH    = os.path.join(DRIVE_ROOT, 'plant_disease_model.keras')

# Verificar archivos en Drive
for label, path in [
    ('manifest.csv',              MANIFEST_PATH),
    ('class_names.json',           CLASSES_PATH),
    ('plant_disease_model.keras',  MODEL_PATH),
]:
    estado = '✅' if os.path.exists(path) else '❌ NO ENCONTRADO'
    print(f'{estado}  {label}')

In [ ]:
import kagglehub

# Descargar los 3 datasets (se cachean; la segunda ejecucion es instantanea)
print('Descargando New Plant Diseases Dataset...')
p_npd = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')
print('Path to dataset files:', p_npd)

print('\nDescargando PlantDEC...')
p_plantdec = kagglehub.dataset_download('andresmgs/plantdec')
print('Path to dataset files:', p_plantdec)

print('\nDescargando PlantSeg...')
p_plantseg = kagglehub.dataset_download('weitianqi/plantseg')
print('Path to dataset files:', p_plantseg)

# Localizar raiz del NPD (estructura anidada)
COLAB_NPD_ROOT = p_npd
for candidate in [
    os.path.join(p_npd, 'New Plant Diseases Dataset(Augmented)', 'New Plant Diseases Dataset(Augmented)'),
    os.path.join(p_npd, 'New Plant Diseases Dataset(Augmented)'),
    p_npd,
]:
    if os.path.isdir(os.path.join(candidate, 'train')):
        COLAB_NPD_ROOT = candidate
        break

COLAB_PLANTSEG_ROOT = p_plantseg
COLAB_PLANTDEC_ROOT = p_plantdec
print(f'\nNPD root: {COLAB_NPD_ROOT}')
print('✅ Datasets listos')

# 🌿 PlantAI — Exploración, Augmentación y Evaluación

Este notebook presenta de forma visual tres aspectos clave del proyecto:

1. **Exploración del dataset** — distribución de clases, fuentes y ejemplos de imágenes
2. **Augmentación de datos** — comparación visual de los 3 niveles (leve · intensa · extrema)
3. **Evaluación del modelo** — accuracy, F1, top-5 y matriz de confusión

---
### Archivos necesarios

Sube los siguientes archivos **directamente** cuando se te solicite en la celda de carga:

| Archivo | Necesario para |
|---|---|
|  | Sección 1 — Exploración |
|  | Sección 1 y 3 |
|  | Sección 3 — Evaluación |
| Imágenes de hoja (/) | Sección 2 — Augmentación (opcional) |

> **Nota:** Puedes subir todos los archivos a la vez en la celda de carga. Las imágenes sueltas se usarán para visualizar la augmentación.

In [ ]:
# Cargar manifest y remap de paths Windows --> Colab
from pathlib import PureWindowsPath

df = pd.read_csv(MANIFEST_PATH)
print(f'Total de imágenes : {len(df):,}')
print(f'Clases únicas     : {df["label"].nunique()}')
print(f'Fuentes           : {list(df["source"].unique())}')
print(f'Splits            : {list(df["split"].unique())}')
print()
print(df.groupby(['source', 'split']).size().unstack(fill_value=0))

# Remap paths: prefijo Windows local --> ruta Kaggle en Colab
source_to_colab = {
    'npd':      globals().get('COLAB_NPD_ROOT'),
    'plantseg': globals().get('COLAB_PLANTSEG_ROOT'),
    'plantdec': globals().get('COLAB_PLANTDEC_ROOT'),
}

source_prefixes = {}
for source in df['source'].unique():
    paths = df[df['source'] == source]['path'].dropna().head(30).tolist()
    parts_list = [PureWindowsPath(p).parts for p in paths]
    common = []
    for level in zip(*parts_list):
        if len(set(level)) == 1:
            common.append(level[0])
        else:
            break
    if common:
        source_prefixes[source] = str(PureWindowsPath(*common))
        print(f'  [{source}] prefijo local: {source_prefixes[source]}')

def remap(row):
    src   = row['source']
    colab = source_to_colab.get(src)
    pfx   = source_prefixes.get(src)
    if not colab or not pfx:
        return row['path']
    try:
        rel = PureWindowsPath(row['path']).relative_to(pfx)
        return os.path.join(colab, str(rel).replace('\\\\', '/').replace('\\', '/'))
    except ValueError:
        return row['path']

df['path'] = df.apply(remap, axis=1)

valid = df['path'].apply(os.path.isfile).sum()
print(f'\nPaths válidos: {valid:,}/{len(df):,}')
if valid > 0:
    print(f'Ejemplo: {df[df["path"].apply(os.path.isfile)].iloc[0]["path"]}')

In [ ]:
# ── Distribución por fuente ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribución del Dataset', fontsize=15, fontweight='bold')

# Tarta: imágenes por fuente
source_counts = df['source'].value_counts()
colores = ['#40916c', '#74c69d', '#f4a261']
axes[0].pie(
    source_counts.values,
    labels=source_counts.index,
    autopct='%1.1f%%',
    colors=colores[:len(source_counts)],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
)
axes[0].set_title('Imágenes por fuente')

# Barras: imágenes por split
split_counts = df['split'].value_counts()
bars = axes[1].bar(split_counts.index, split_counts.values, color=['#40916c', '#74c69d', '#d8f3dc'], edgecolor='white')
axes[1].set_title('Imágenes por split')
axes[1].set_ylabel('Cantidad')
for bar in bars:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 100,
        f'{bar.get_height():,}',
        ha='center', fontsize=11, fontweight='bold',
    )

plt.tight_layout()
plt.show()

In [ ]:
# ── Top-20 clases por número de imágenes ──────────────────────────────────────
class_counts = df['label'].value_counts()
top20 = class_counts.head(20)

# Nombres cortos para el eje
short = [l.replace('_', ' ').replace('(', '').replace(')', '') for l in top20.index]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(short[::-1], top20.values[::-1], color='#40916c', edgecolor='white')
ax.set_xlabel('Número de imágenes')
ax.set_title('Top 20 clases por cantidad de imágenes', fontsize=13, fontweight='bold')
for bar in bars:
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
ax.set_xlim(0, top20.values.max() * 1.12)
plt.tight_layout()
plt.show()

print(f'\nMínimo de imágenes por clase : {class_counts.min():,}')
print(f'Máximo de imágenes por clase : {class_counts.max():,}')
print(f'Promedio                     : {class_counts.mean():.0f}')

In [ ]:
# ── Distribución completa de clases (todas las 114) ───────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
sorted_counts = class_counts.sort_values(ascending=True)
short_all = [l.split('___')[-1].replace('_', ' ')[:22] for l in sorted_counts.index]

# Color: verde = saludable, naranja = enfermedad
colors = ['#74c69d' if 'healthy' in lbl else '#f4a261' for lbl in sorted_counts.index]

ax.barh(range(len(sorted_counts)), sorted_counts.values, color=colors, edgecolor='none', height=0.8)
ax.set_yticks(range(len(sorted_counts)))
ax.set_yticklabels(short_all, fontsize=6)
ax.set_xlabel('Número de imágenes')
ax.set_title('Distribución de las 114 clases\n(verde = saludable, naranja = enfermedad)', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#74c69d', label='Saludable'),
    Patch(color='#f4a261', label='Enfermedad'),
], loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Muestras visuales del dataset ─────────────────────────────────────────────
# Muestra 3 imágenes de cada fuente disponible
N_PER_SOURCE = 3
sources = df['source'].unique()

fig, axes = plt.subplots(len(sources), N_PER_SOURCE, figsize=(N_PER_SOURCE * 4, len(sources) * 4))
if len(sources) == 1:
    axes = axes.reshape(1, -1)

fig.suptitle('Muestras del dataset por fuente', fontsize=14, fontweight='bold', y=1.01)

for row_idx, source in enumerate(sources):
    subset = df[df['source'] == source].sample(n=N_PER_SOURCE, random_state=42)
    for col_idx, (_, row) in enumerate(subset.iterrows()):
        ax = axes[row_idx, col_idx]
        try:
            img = Image.open(row['path']).convert('RGB').resize((224, 224))
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'imagen\nno disponible', ha='center', va='center', transform=ax.transAxes)
            ax.set_facecolor('#f0f0f0')

        label_short = row['label'].replace('_', ' ').replace('(including sour)', '')
        ax.set_title(f'{label_short}\n[{source}]', fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.show()

# 🌿 PlantAI — Exploración, Augmentación y Evaluación

Este notebook presenta de forma visual tres aspectos clave del proyecto:

1. **Exploración del dataset** — distribución de clases, fuentes y ejemplos de imágenes
2. **Augmentación de datos** — comparación visual de los 3 niveles (leve · intensa · extrema)
3. **Evaluación del modelo** — accuracy, F1, top-5 y matriz de confusión

---
### Archivos necesarios

Sube los siguientes archivos **directamente** cuando se te solicite en la celda de carga:

| Archivo | Necesario para |
|---|---|
|  | Sección 1 — Exploración |
|  | Sección 1 y 3 |
|  | Sección 3 — Evaluación |
| Imágenes de hoja (/) | Sección 2 — Augmentación (opcional) |

> **Nota:** Puedes subir todos los archivos a la vez en la celda de carga. Las imágenes sueltas se usarán para visualizar la augmentación.

In [ ]:
# ── Funciones de augmentación (replicadas del proyecto) ───────────────────────
IMG_SIZE = 224

@tf.function
def augment_light(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + 20, IMG_SIZE + 20)
    image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, 3])
    return tf.clip_by_value(image, 0.0, 1.0), label

@tf.function
def augment_heavy(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    pad = tf.random.uniform([], minval=30, maxval=50, dtype=tf.int32)
    image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + pad, IMG_SIZE + pad)
    image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, 3])
    image = tf.image.random_brightness(image, max_delta=0.25)
    image = tf.image.random_contrast(image, lower=0.6, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.5, upper=1.5)
    image = tf.image.random_hue(image, max_delta=0.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    s = tf.random.uniform([], 0.5, 0.88)
    si = tf.cast(tf.cast(IMG_SIZE, tf.float32) * s, tf.int32)
    blurred = tf.image.resize(tf.image.resize(image, [si, si]), [IMG_SIZE, IMG_SIZE])
    do_blur = tf.cast(tf.random.uniform([]) > 0.5, tf.float32)
    image = do_blur * blurred + (1.0 - do_blur) * image
    noise = tf.random.normal(tf.shape(image), stddev=0.025)
    return tf.clip_by_value(image + noise, 0.0, 1.0), label

@tf.function
def augment_extreme(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    pad = tf.random.uniform([], minval=50, maxval=70, dtype=tf.int32)
    image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + pad, IMG_SIZE + pad)
    image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, 3])
    image = tf.image.random_brightness(image, max_delta=0.35)
    image = tf.image.random_contrast(image, lower=0.4, upper=1.6)
    image = tf.image.random_saturation(image, lower=0.3, upper=1.7)
    image = tf.image.random_hue(image, max_delta=0.15)
    image = tf.clip_by_value(image, 0.0, 1.0)
    gray = tf.tile(tf.image.rgb_to_grayscale(image), [1, 1, 3])
    image = tf.cast(tf.random.uniform([]) > 0.88, tf.float32) * gray + \
            tf.cast(tf.random.uniform([]) <= 0.88, tf.float32) * image
    s = tf.random.uniform([], 0.35, 0.78)
    si = tf.cast(tf.cast(IMG_SIZE, tf.float32) * s, tf.int32)
    blurred = tf.image.resize(tf.image.resize(image, [si, si]), [IMG_SIZE, IMG_SIZE])
    do_blur = tf.cast(tf.random.uniform([]) > 0.35, tf.float32)
    image = do_blur * blurred + (1.0 - do_blur) * image
    noise = tf.random.normal(tf.shape(image), stddev=0.045)
    image = tf.clip_by_value(image + noise, 0.0, 1.0)
    patch_size = tf.random.uniform([], 20, 50, dtype=tf.int32)
    y0 = tf.random.uniform([], 0, IMG_SIZE - patch_size, dtype=tf.int32)
    x0 = tf.random.uniform([], 0, IMG_SIZE - patch_size, dtype=tf.int32)
    row_in = tf.cast((tf.range(IMG_SIZE) >= y0) & (tf.range(IMG_SIZE) < y0 + patch_size), tf.float32)
    col_in = tf.cast((tf.range(IMG_SIZE) >= x0) & (tf.range(IMG_SIZE) < x0 + patch_size), tf.float32)
    mask = tf.tile(tf.expand_dims(tf.tensordot(row_in, col_in, axes=0), -1), [1, 1, 3])
    do_cut = tf.cast(tf.random.uniform([]) > 0.5, tf.float32)
    image = image * (1.0 - do_cut * mask) + 0.5 * do_cut * mask
    return tf.clip_by_value(image, 0.0, 1.0), label

def load_image_tensor(path):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32) / 255.0

print('Funciones de augmentación definidas ✅')

In [ ]:
# Seleccionar imagenes de muestra del dataset NPD descargado
import glob as glob_module
from pathlib import Path
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

sample_paths = []

# Buscar imagenes en el directorio train del dataset NPD
if globals().get('COLAB_NPD_ROOT'):
    all_imgs = glob_module.glob(
        os.path.join(COLAB_NPD_ROOT, 'train', '**', '*.jpg'), recursive=True
    )
    if all_imgs:
        random.shuffle(all_imgs)
        sample_paths = all_imgs[:8]

# Fallback: usar paths validos del manifest
if not sample_paths:
    valid_df = df[df['path'].apply(os.path.isfile)]
    if len(valid_df) > 0:
        sample_rows = valid_df.groupby('label').first().sample(
            min(4, valid_df['label'].nunique()), random_state=7
        )
        sample_paths = sample_rows['path'].tolist()

if not sample_paths:
    print('⚠️  No se encontraron imagenes. Verifica que los datasets se descargaron.')
else:
    print(f'Imagenes de muestra: {len(sample_paths)}')
    for p in sample_paths[:4]:
        print(' ', os.path.basename(p))

In [ ]:
# ── Visualización comparativa de los 3 niveles de augmentación ────────────────
N_VARIANTS = 4   # cuántas versiones augmentadas mostrar por nivel
augment_fns   = [augment_light, augment_heavy, augment_extreme]
augment_names = ['Original', 'Leve (Fase 1)', 'Intensa (Fase 2)', 'Extrema (Fase 3)']
augment_colors = ['#888888', '#40916c', '#f4a261', '#e76f51']

# Usar la primera imagen disponible
if not sample_paths:
    print('⚠️  No hay imágenes de muestra disponibles. Sube imágenes a sample_images/ en Drive.')
else:
    src_path = sample_paths[0]
    img_orig = load_image_tensor(src_path)

    n_cols = N_VARIANTS
    n_rows = len(augment_fns) + 1  # +1 para la fila original
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3.2))
    fig.suptitle('Comparación de niveles de augmentación', fontsize=15, fontweight='bold', y=1.01)

    for col in range(n_cols):
        axes[0, col].imshow(img_orig.numpy())
        axes[0, col].set_title('Original' if col == 0 else '', fontsize=9)
        axes[0, col].axis('off')

    for row, (fn, name, color) in enumerate(zip(augment_fns, augment_names[1:], augment_colors[1:]), start=1):
        for col in range(n_cols):
            aug_img, _ = fn(img_orig, 0)
            axes[row, col].imshow(aug_img.numpy())
            if col == 0:
                axes[row, col].set_ylabel(name, fontsize=10, color=color, fontweight='bold', rotation=0,
                                          labelpad=80, va='center')
            axes[row, col].set_title(f'v{col+1}', fontsize=8)
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Comparación en cuadrícula para múltiples imágenes ─────────────────────────
if len(sample_paths) >= 2:
    imgs_to_show = sample_paths[:min(4, len(sample_paths))]
    fns_all      = [None, augment_light, augment_heavy, augment_extreme]
    fns_names    = ['Original', 'Leve', 'Intensa', 'Extrema']
    fns_colors   = ['#555', '#40916c', '#f4a261', '#e76f51']

    fig, axes = plt.subplots(len(imgs_to_show), 4, figsize=(16, len(imgs_to_show) * 4))
    if len(imgs_to_show) == 1:
        axes = axes.reshape(1, -1)

    for r, path in enumerate(imgs_to_show):
        base = load_image_tensor(path)
        for c, (fn, name, color) in enumerate(zip(fns_all, fns_names, fns_colors)):
            img = base if fn is None else fn(base, 0)[0]
            axes[r, c].imshow(img.numpy())
            axes[r, c].axis('off')
            if r == 0:
                axes[r, c].set_title(name, fontsize=12, color=color, fontweight='bold')

    fig.suptitle('Augmentación en múltiples imágenes', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

# 🌿 PlantAI — Exploración, Augmentación y Evaluación

Este notebook presenta de forma visual tres aspectos clave del proyecto:

1. **Exploración del dataset** — distribución de clases, fuentes y ejemplos de imágenes
2. **Augmentación de datos** — comparación visual de los 3 niveles (leve · intensa · extrema)
3. **Evaluación del modelo** — accuracy, F1, top-5 y matriz de confusión

---
### Archivos necesarios

Sube los siguientes archivos **directamente** cuando se te solicite en la celda de carga:

| Archivo | Necesario para |
|---|---|
|  | Sección 1 — Exploración |
|  | Sección 1 y 3 |
|  | Sección 3 — Evaluación |
| Imágenes de hoja (/) | Sección 2 — Augmentación (opcional) |

> **Nota:** Puedes subir todos los archivos a la vez en la celda de carga. Las imágenes sueltas se usarán para visualizar la augmentación.

In [ ]:
# Cargar modelo y nombres de clases
if not os.path.exists(MODEL_PATH):
    print('❌ Modelo no encontrado. Sube plant_disease_model.keras a Drive.')
    model = None
else:
    print('Cargando modelo…')
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f'✅ Modelo cargado — {model.count_params():,} parámetros')

if os.path.exists(CLASSES_PATH):
    with open(CLASSES_PATH, encoding='utf-8') as f:
        CLASS_NAMES = json.load(f)
    print(f'✅ {len(CLASS_NAMES)} clases cargadas')
else:
    CLASS_NAMES = None
    print('❌ class_names.json no encontrado.')

In [ ]:
# ── Evaluar sobre imágenes del manifest (val split) ───────────────────────────
if model is None or CLASS_NAMES is None:
    print('Omitiendo evaluación — falta modelo o clases.')
else:
    class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

    val_df = df[(df['split'] == 'val') & df['path'].apply(os.path.isfile)]
    val_df = val_df[val_df['label'].isin(class_to_idx)]

    MAX_EVAL = 2000  # límite para no tardar demasiado en Colab
    if len(val_df) > MAX_EVAL:
        val_df = val_df.groupby('label', group_keys=False).apply(
            lambda x: x.sample(min(len(x), max(1, MAX_EVAL // val_df['label'].nunique())), random_state=42)
        ).reset_index(drop=True)

    print(f'Evaluando sobre {len(val_df):,} imágenes de validación…')

    all_proba, all_true = [], []
    BATCH = 64

    for start in range(0, len(val_df), BATCH):
        batch = val_df.iloc[start:start+BATCH]
        imgs = []
        for path in batch['path']:
            try:
                img = load_image_tensor(path).numpy()
                imgs.append(img)
            except Exception:
                imgs.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32))
        imgs_arr = np.array(imgs)
        proba = model.predict(imgs_arr, verbose=0)
        all_proba.extend(proba)
        all_true.extend([class_to_idx[l] for l in batch['label']])

        if (start // BATCH + 1) % 5 == 0:
            print(f'  {start + BATCH}/{len(val_df)} imágenes procesadas…')

    all_proba = np.array(all_proba)
    all_true  = np.array(all_true, dtype=int)
    all_pred  = np.argmax(all_proba, axis=1)
    print('\nInferencia completada ✅')

In [ ]:
# ── Métricas globales ─────────────────────────────────────────────────────────
if model is not None and CLASS_NAMES is not None:
    from sklearn.metrics import f1_score, top_k_accuracy_score

    acc      = float(np.mean(all_pred == all_true))
    top5_k   = min(5, len(CLASS_NAMES))
    top5_acc = float(top_k_accuracy_score(all_true, all_proba, k=top5_k))
    f1_mac   = float(f1_score(all_true, all_pred, average='macro',    zero_division=0))
    f1_w     = float(f1_score(all_true, all_pred, average='weighted', zero_division=0))

    # Resumen visual
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.axis('off')
    metrics_data = [
        ['Accuracy Top-1',        f'{acc*100:.2f}%'],
        [f'Accuracy Top-{top5_k}', f'{top5_acc*100:.2f}%'],
        ['F1 Macro',              f'{f1_mac:.4f}'],
        ['F1 Ponderado',          f'{f1_w:.4f}'],
        ['Imágenes evaluadas',    f'{len(all_true):,}'],
        ['Clases presentes',      f'{len(set(all_true.tolist()))}'],
    ]
    tbl = ax.table(
        cellText=metrics_data,
        colLabels=['Métrica', 'Valor'],
        cellLoc='center',
        loc='center',
        colWidths=[0.5, 0.3],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(13)
    tbl.scale(1, 2.2)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#1b4332')
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#d8f3dc')

    ax.set_title('Resultados de Evaluación', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Barras de F1 por clase (top-30 y bottom-30) ───────────────────────────────
if model is not None and CLASS_NAMES is not None:
    from sklearn.metrics import f1_score as f1_per_class

    present_idx  = sorted(set(all_true.tolist()))
    present_names = [CLASS_NAMES[i] for i in present_idx]
    f1_per       = f1_score(all_true, all_pred, labels=present_idx, average=None, zero_division=0)

    f1_df = pd.DataFrame({'clase': present_names, 'f1': f1_per})
    f1_df['short'] = f1_df['clase'].apply(lambda x: x.split('___')[-1].replace('_', ' ')[:28])
    f1_df['healthy'] = f1_df['clase'].str.contains('healthy')

    fig, axes = plt.subplots(1, 2, figsize=(18, 10))

    for ax, top, title in [
        (axes[0], f1_df.nlargest(30, 'f1'),  'Top 30 clases — mayor F1'),
        (axes[1], f1_df.nsmallest(30, 'f1'), 'Bottom 30 clases — menor F1'),
    ]:
        colors = ['#74c69d' if h else '#f4a261' for h in top['healthy']]
        bars = ax.barh(top['short'], top['f1'], color=colors, edgecolor='white')
        ax.set_xlim(0, 1.05)
        ax.set_xlabel('F1-score')
        ax.set_title(title, fontsize=12, fontweight='bold')
        for bar in bars:
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{bar.get_width():.2f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Predicciones de muestra con confianza ─────────────────────────────────────
TRADUCCIONES = {
    'Apple___Apple_scab': 'Manzana — Sarna',
    'Apple___healthy': 'Manzana — Saludable',
    'Tomato___Early_blight': 'Tomate — Tizón temprano',
    'Tomato___Late_blight': 'Tomate — Tizón tardío',
    'Tomato___healthy': 'Tomate — Saludable',
    'Potato___Early_blight': 'Papa — Tizón temprano',
    'Potato___Late_blight': 'Papa — Tizón tardío',
    'Grape___Black_rot': 'Uva — Pudrición negra',
    'Corn_(maize)___Common_rust_': 'Maíz — Roya común',
}

def label_es(raw):
    if raw in TRADUCCIONES:
        return TRADUCCIONES[raw]
    parts = raw.split('___')
    return f"{parts[0]} — {parts[1].replace('_', ' ')}" if len(parts) == 2 else raw

if model is not None and CLASS_NAMES is not None:
    N_SHOW = 12
    # Mezcla de aciertos y errores
    correct_idx = np.where(all_pred == all_true)[0]
    wrong_idx   = np.where(all_pred != all_true)[0]

    sample_correct = np.random.choice(correct_idx, min(N_SHOW // 2, len(correct_idx)), replace=False)
    sample_wrong   = np.random.choice(wrong_idx,   min(N_SHOW // 2, len(wrong_idx)),   replace=False)
    sample_idx     = np.concatenate([sample_correct, sample_wrong])
    np.random.shuffle(sample_idx)

    val_paths_list = val_df['path'].tolist()

    cols = 4
    rows = len(sample_idx) // cols + (1 if len(sample_idx) % cols else 0)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    for ax_i, idx in enumerate(sample_idx):
        ax = axes[ax_i]
        try:
            img = Image.open(val_paths_list[idx]).convert('RGB').resize((224, 224))
            ax.imshow(img)
        except Exception:
            ax.set_facecolor('#eee')

        pred_name = CLASS_NAMES[all_pred[idx]]
        true_name = CLASS_NAMES[all_true[idx]]
        conf      = float(all_proba[idx, all_pred[idx]])
        correct   = all_pred[idx] == all_true[idx]

        color = '#1b4332' if correct else '#c0392b'
        mark  = '✅' if correct else '❌'
        ax.set_title(
            f'{mark} {label_es(pred_name)}\n({conf*100:.1f}%)\nReal: {label_es(true_name)}',
            fontsize=8, color=color,
        )
        ax.axis('off')

    for ax in axes[len(sample_idx):]:
        ax.axis('off')

    fig.suptitle('Muestras de predicción (verde = correcto, rojo = error)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 📁 Exportar resultados

Guarda las métricas finales como JSON en Google Drive.

In [ ]:
if model is not None and CLASS_NAMES is not None:
    results = {
        'accuracy_top1':  round(acc, 4),
        'accuracy_top5':  round(top5_acc, 4),
        'f1_macro':       round(f1_mac, 4),
        'f1_weighted':    round(f1_w, 4),
        'n_samples':      int(len(all_true)),
        'n_classes':      int(len(present_idx)),
    }

    out = os.path.join(DRIVE_ROOT, 'eval_metrics_colab.json')
    with open(out, 'w') as f:
        json.dump(results, f, indent=2)

    print('Métricas guardadas en Drive:')
    for k, v in results.items():
        print(f'  {k:<20s} {v}')